In [1]:
from __future__ import annotations

import os
from datetime import UTC, datetime
from getpass import getpass
from typing import Any

import weaviate
import weaviate.classes as wvc

from weaviate.classes.init import Auth
from weaviate.classes.query import Filter, MetadataQuery

In [2]:
if "WEAVIATE_URL" not in os.environ:
    os.environ["WEAVIATE_URL"] = getpass("WEAVIATE_URL: ")

if "WEAVIATE_API_KEY" not in os.environ:
    os.environ["WEAVIATE_API_KEY"] = getpass("WEAVIATE_API_KEY: ")

if "OPENAI_API_KEY" not in os.environ:
    os.environ["OPENAI_API_KEY"] = getpass("OPENAI_API_KEY: ")

In [3]:
client = weaviate.connect_to_weaviate_cloud(
    cluster_url=os.environ["WEAVIATE_URL"],
    auth_credentials=Auth.api_key(os.environ["WEAVIATE_API_KEY"]),
    headers={
        "X-OpenAI-Api-Key": os.environ["OPENAI_API_KEY"],
    },
)

print("Client ready:", client.is_ready())

Client ready: True


In [4]:
def dt(year: int, month: int, day: int) -> datetime:
    return datetime(year, month, day, tzinfo=UTC)

In [5]:
COLLECTION_NAME = "RankedKnowledgeArticle"

if client.collections.exists(COLLECTION_NAME):
    client.collections.delete(COLLECTION_NAME)

articles = client.collections.create(
    name=COLLECTION_NAME,
    vector_config=wvc.config.Configure.Vectors.text2vec_openai(
        model="text-embedding-3-small",
    ),
    generative_config=wvc.config.Configure.Generative.openai(
        model="gpt-4o-mini",
    ),
    properties=[
        wvc.config.Property(
            name="title",
            data_type=wvc.config.DataType.TEXT,
        ),
        wvc.config.Property(
            name="content",
            data_type=wvc.config.DataType.TEXT,
        ),
        wvc.config.Property(
            name="topic",
            data_type=wvc.config.DataType.TEXT,
        ),
        wvc.config.Property(
            name="difficulty",
            data_type=wvc.config.DataType.TEXT,
        ),
        wvc.config.Property(
            name="priority",
            data_type=wvc.config.DataType.INT,
        ),
        wvc.config.Property(
            name="reading_time_minutes",
            data_type=wvc.config.DataType.INT,
        ),
        wvc.config.Property(
            name="quality_score",
            data_type=wvc.config.DataType.NUMBER,
        ),
        wvc.config.Property(
            name="production_ready",
            data_type=wvc.config.DataType.BOOL,
        ),
        wvc.config.Property(
            name="updated_at",
            data_type=wvc.config.DataType.DATE,
        ),
    ],
)

print("Created collection:", COLLECTION_NAME)

Created collection: RankedKnowledgeArticle


In [6]:
article_data = [
    {
        "title": "Weaviate Cloud Authentication",
        "content": (
            "Connect to Weaviate Cloud using a cluster URL, Weaviate API key "
            "and provider headers such as the OpenAI API key."
        ),
        "topic": "cloud",
        "difficulty": "beginner",
        "priority": 5,
        "reading_time_minutes": 8,
        "quality_score": 9.2,
        "production_ready": True,
        "updated_at": dt(2026, 5, 20),
    },
    {
        "title": "Vector Search Basics",
        "content": (
            "Vector search uses embeddings to find semantically similar objects. "
            "It is useful when users describe meaning instead of exact keywords."
        ),
        "topic": "vector_search",
        "difficulty": "beginner",
        "priority": 4,
        "reading_time_minutes": 10,
        "quality_score": 8.8,
        "production_ready": True,
        "updated_at": dt(2026, 5, 12),
    },
    {
        "title": "BM25 Keyword Search",
        "content": (
            "BM25 is useful for exact keywords, technical terms, product names, "
            "categories and identifiers."
        ),
        "topic": "bm25",
        "difficulty": "intermediate",
        "priority": 3,
        "reading_time_minutes": 12,
        "quality_score": 8.4,
        "production_ready": True,
        "updated_at": dt(2026, 5, 10),
    },
    {
        "title": "Hybrid Search with Alpha",
        "content": (
            "Hybrid search combines BM25 keyword search with semantic vector search. "
            "The alpha parameter controls the balance between lexical and semantic ranking."
        ),
        "topic": "hybrid_search",
        "difficulty": "intermediate",
        "priority": 5,
        "reading_time_minutes": 15,
        "quality_score": 9.3,
        "production_ready": True,
        "updated_at": dt(2026, 5, 18),
    },
    {
        "title": "Generative Search",
        "content": (
            "Generative search retrieves relevant objects and uses a language model "
            "to generate answers with single_prompt or grouped_task."
        ),
        "topic": "generative_search",
        "difficulty": "intermediate",
        "priority": 5,
        "reading_time_minutes": 18,
        "quality_score": 9.1,
        "production_ready": True,
        "updated_at": dt(2026, 5, 19),
    },
    {
        "title": "Focused RAG over Notebooks",
        "content": (
            "Focused RAG works best when the collection contains clean chunks from "
            "relevant notebooks and Markdown concept notes instead of unrelated files."
        ),
        "topic": "rag",
        "difficulty": "advanced",
        "priority": 5,
        "reading_time_minutes": 25,
        "quality_score": 9.7,
        "production_ready": True,
        "updated_at": dt(2026, 5, 23),
    },
    {
        "title": "Named Vectors",
        "content": (
            "Named vectors allow one object to have multiple vector spaces, "
            "for example title_vector and description_vector."
        ),
        "topic": "named_vectors",
        "difficulty": "advanced",
        "priority": 4,
        "reading_time_minutes": 20,
        "quality_score": 9.0,
        "production_ready": True,
        "updated_at": dt(2026, 5, 17),
    },
    {
        "title": "Cross-References",
        "content": (
            "Cross-references connect objects between collections using reference properties, "
            "for example reviews connected to products."
        ),
        "topic": "cross_references",
        "difficulty": "advanced",
        "priority": 3,
        "reading_time_minutes": 22,
        "quality_score": 8.6,
        "production_ready": False,
        "updated_at": dt(2026, 5, 14),
    },
    {
        "title": "Multi-Tenancy",
        "content": (
            "Multi-tenancy isolates data for multiple clients, companies or users "
            "inside one shared collection schema."
        ),
        "topic": "multi_tenancy",
        "difficulty": "advanced",
        "priority": 5,
        "reading_time_minutes": 21,
        "quality_score": 9.4,
        "production_ready": True,
        "updated_at": dt(2026, 5, 21),
    },
    {
        "title": "JSONL Snapshot Export",
        "content": (
            "A JSONL snapshot stores one JSON object per line and can be used for "
            "application-level export, inspection and restore workflows."
        ),
        "topic": "snapshot",
        "difficulty": "intermediate",
        "priority": 3,
        "reading_time_minutes": 16,
        "quality_score": 8.7,
        "production_ready": True,
        "updated_at": dt(2026, 5, 16),
    },
    {
        "title": "Geo Search for EV Stations",
        "content": (
            "Geo search combines location filtering with semantic search, metadata filters "
            "and recommendation logic for local applications."
        ),
        "topic": "geo_search",
        "difficulty": "advanced",
        "priority": 4,
        "reading_time_minutes": 24,
        "quality_score": 9.0,
        "production_ready": True,
        "updated_at": dt(2026, 5, 24),
    },
]

In [7]:
result = articles.data.insert_many(article_data)

if result.errors:
    print("Import errors:")
    print(result.errors)
else:
    print("Import completed.")

Import completed.


In [8]:
response = articles.query.fetch_objects(
    limit=20,
    return_properties=[
        "title",
        "topic",
        "difficulty",
        "priority",
        "reading_time_minutes",
        "quality_score",
        "production_ready",
        "updated_at",
    ],
)

for obj in response.objects:
    print(obj.properties)
    print("-" * 80)

{'quality_score': 8.4, 'title': 'BM25 Keyword Search', 'production_ready': True, 'priority': 3, 'difficulty': 'intermediate', 'reading_time_minutes': 12, 'updated_at': datetime.datetime(2026, 5, 10, 0, 0, tzinfo=datetime.timezone.utc), 'topic': 'bm25'}
--------------------------------------------------------------------------------
{'quality_score': 9.0, 'title': 'Geo Search for EV Stations', 'production_ready': True, 'priority': 4, 'difficulty': 'advanced', 'reading_time_minutes': 24, 'topic': 'geo_search', 'updated_at': datetime.datetime(2026, 5, 24, 0, 0, tzinfo=datetime.timezone.utc)}
--------------------------------------------------------------------------------
{'quality_score': 8.7, 'title': 'JSONL Snapshot Export', 'production_ready': True, 'priority': 3, 'difficulty': 'intermediate', 'reading_time_minutes': 16, 'updated_at': datetime.datetime(2026, 5, 16, 0, 0, tzinfo=datetime.timezone.utc), 'topic': 'snapshot'}
----------------------------------------------------------------

In [9]:
response = articles.query.near_text(
    query="best Weaviate concepts for building production RAG applications",
    limit=8,
    return_properties=[
        "title",
        "content",
        "topic",
        "difficulty",
        "priority",
        "reading_time_minutes",
        "quality_score",
        "production_ready",
        "updated_at",
    ],
    return_metadata=MetadataQuery(distance=True),
)

for obj in response.objects:
    print("Distance:", obj.metadata.distance)
    print("Title:", obj.properties["title"])
    print("Topic:", obj.properties["topic"])
    print("Priority:", obj.properties["priority"])
    print("Quality:", obj.properties["quality_score"])
    print("Production ready:", obj.properties["production_ready"])
    print("Reading time:", obj.properties["reading_time_minutes"])
    print("Updated at:", obj.properties["updated_at"])
    print("-" * 80)

Distance: 0.5503439903259277
Title: Focused RAG over Notebooks
Topic: rag
Priority: 5
Quality: 9.7
Production ready: True
Reading time: 25
Updated at: 2026-05-23 00:00:00+00:00
--------------------------------------------------------------------------------
Distance: 0.5901618003845215
Title: Weaviate Cloud Authentication
Topic: cloud
Priority: 5
Quality: 9.2
Production ready: True
Reading time: 8
Updated at: 2026-05-20 00:00:00+00:00
--------------------------------------------------------------------------------
Distance: 0.7292242646217346
Title: Multi-Tenancy
Topic: multi_tenancy
Priority: 5
Quality: 9.4
Production ready: True
Reading time: 21
Updated at: 2026-05-21 00:00:00+00:00
--------------------------------------------------------------------------------
Distance: 0.7475579977035522
Title: Cross-References
Topic: cross_references
Priority: 3
Quality: 8.6
Production ready: False
Reading time: 22
Updated at: 2026-05-14 00:00:00+00:00
--------------------------------------------

In [10]:
def parse_weaviate_datetime(value: Any) -> datetime:
    if isinstance(value, datetime):
        return value

    if isinstance(value, str):
        return datetime.fromisoformat(value.replace("Z", "+00:00"))

    raise TypeError(f"Unsupported datetime value: {value!r}")

In [11]:
def freshness_score(updated_at: Any, *, reference_date: datetime) -> float:
    parsed_date = parse_weaviate_datetime(updated_at)
    age_days = (reference_date - parsed_date).days

    if age_days <= 3:
        return 1.0

    if age_days <= 7:
        return 0.8

    if age_days <= 14:
        return 0.6

    if age_days <= 30:
        return 0.4

    return 0.2

In [12]:
def rerank_articles(objects, *, reference_date: datetime) -> list[dict[str, Any]]:
    ranked = []

    for obj in objects:
        props = obj.properties

        distance = obj.metadata.distance
        semantic_score = max(0.0, 1.0 - distance)

        priority_score = props["priority"] / 5
        quality_score = props["quality_score"] / 10
        production_score = 1.0 if props["production_ready"] else 0.0
        freshness = freshness_score(
            props["updated_at"],
            reference_date=reference_date,
        )

        reading_time = props["reading_time_minutes"]

        if reading_time <= 15:
            reading_time_score = 1.0
        elif reading_time <= 25:
            reading_time_score = 0.7
        else:
            reading_time_score = 0.4

        final_score = (
            semantic_score * 0.45
            + priority_score * 0.20
            + quality_score * 0.15
            + production_score * 0.10
            + freshness * 0.05
            + reading_time_score * 0.05
        )

        ranked.append(
            {
                "uuid": obj.uuid,
                "title": props["title"],
                "content": props["content"],
                "topic": props["topic"],
                "difficulty": props["difficulty"],
                "priority": props["priority"],
                "reading_time_minutes": props["reading_time_minutes"],
                "quality_score": props["quality_score"],
                "production_ready": props["production_ready"],
                "updated_at": props["updated_at"],
                "distance": distance,
                "semantic_score": round(semantic_score, 4),
                "rerank_score": round(final_score, 4),
            }
        )

    return sorted(
        ranked,
        key=lambda item: item["rerank_score"],
        reverse=True,
    )

In [13]:
reference_date = dt(2026, 5, 25)

ranked_articles = rerank_articles(
    response.objects,
    reference_date=reference_date,
)

for item in ranked_articles:
    print("Rerank score:", item["rerank_score"])
    print("Semantic score:", item["semantic_score"])
    print("Distance:", item["distance"])
    print("Title:", item["title"])
    print("Topic:", item["topic"])
    print("Priority:", item["priority"])
    print("Quality:", item["quality_score"])
    print("Production ready:", item["production_ready"])
    print("Reading time:", item["reading_time_minutes"])
    print("Updated at:", item["updated_at"])
    print("-" * 80)

Rerank score: 0.7328
Semantic score: 0.4497
Distance: 0.5503439903259277
Title: Focused RAG over Notebooks
Topic: rag
Priority: 5
Quality: 9.7
Production ready: True
Reading time: 25
Updated at: 2026-05-23 00:00:00+00:00
--------------------------------------------------------------------------------
Rerank score: 0.7124
Semantic score: 0.4098
Distance: 0.5901618003845215
Title: Weaviate Cloud Authentication
Topic: cloud
Priority: 5
Quality: 9.2
Production ready: True
Reading time: 8
Updated at: 2026-05-20 00:00:00+00:00
--------------------------------------------------------------------------------
Rerank score: 0.6378
Semantic score: 0.2708
Distance: 0.7292242646217346
Title: Multi-Tenancy
Topic: multi_tenancy
Priority: 5
Quality: 9.4
Production ready: True
Reading time: 21
Updated at: 2026-05-21 00:00:00+00:00
--------------------------------------------------------------------------------
Rerank score: 0.6168
Semantic score: 0.234
Distance: 0.7660419940948486
Title: Generative Sea

In [14]:
print("Original Weaviate order")
print("-" * 80)

for number, obj in enumerate(response.objects, start=1):
    print(number, obj.properties["title"], "| distance:", obj.metadata.distance)

print("\nReranked order")
print("-" * 80)

for number, item in enumerate(ranked_articles, start=1):
    print(number, item["title"], "| rerank score:", item["rerank_score"])

Original Weaviate order
--------------------------------------------------------------------------------
1 Focused RAG over Notebooks | distance: 0.5503439903259277
2 Weaviate Cloud Authentication | distance: 0.5901618003845215
3 Multi-Tenancy | distance: 0.7292242646217346
4 Cross-References | distance: 0.7475579977035522
5 Named Vectors | distance: 0.7513396739959717
6 JSONL Snapshot Export | distance: 0.7592060565948486
7 Generative Search | distance: 0.7660419940948486
8 Vector Search Basics | distance: 0.767974853515625

Reranked order
--------------------------------------------------------------------------------
1 Focused RAG over Notebooks | rerank score: 0.7328
2 Weaviate Cloud Authentication | rerank score: 0.7124
3 Multi-Tenancy | rerank score: 0.6378
4 Generative Search | rerank score: 0.6168
5 Vector Search Basics | rerank score: 0.5764
6 Named Vectors | rerank score: 0.5719
7 JSONL Snapshot Export | rerank score: 0.5239
8 Cross-References | rerank score: 0.4276


In [15]:
production_ranked = [
    item
    for item in ranked_articles
    if item["production_ready"]
]

for item in production_ranked:
    print("Rerank score:", item["rerank_score"])
    print("Title:", item["title"])
    print("Topic:", item["topic"])
    print("Production ready:", item["production_ready"])
    print("-" * 80)

Rerank score: 0.7328
Title: Focused RAG over Notebooks
Topic: rag
Production ready: True
--------------------------------------------------------------------------------
Rerank score: 0.7124
Title: Weaviate Cloud Authentication
Topic: cloud
Production ready: True
--------------------------------------------------------------------------------
Rerank score: 0.6378
Title: Multi-Tenancy
Topic: multi_tenancy
Production ready: True
--------------------------------------------------------------------------------
Rerank score: 0.6168
Title: Generative Search
Topic: generative_search
Production ready: True
--------------------------------------------------------------------------------
Rerank score: 0.5764
Title: Vector Search Basics
Topic: vector_search
Production ready: True
--------------------------------------------------------------------------------
Rerank score: 0.5719
Title: Named Vectors
Topic: named_vectors
Production ready: True
-----------------------------------------------------

In [16]:
def two_stage_search(
    query: str,
    *,
    candidate_limit: int = 8,
    final_limit: int = 3,
    only_production_ready: bool = True,
    reference_date: datetime,
) -> list[dict[str, Any]]:
    response = articles.query.near_text(
        query=query,
        limit=candidate_limit,
        return_properties=[
            "title",
            "content",
            "topic",
            "difficulty",
            "priority",
            "reading_time_minutes",
            "quality_score",
            "production_ready",
            "updated_at",
        ],
        return_metadata=MetadataQuery(distance=True),
    )

    ranked = rerank_articles(
        response.objects,
        reference_date=reference_date,
    )

    if only_production_ready:
        ranked = [
            item
            for item in ranked
            if item["production_ready"]
        ]

    return ranked[:final_limit]

In [17]:
results = two_stage_search(
    "Weaviate features for production RAG and SaaS applications",
    candidate_limit=10,
    final_limit=5,
    only_production_ready=True,
    reference_date=dt(2026, 5, 25),
)

for item in results:
    print("Score:", item["rerank_score"])
    print("Title:", item["title"])
    print("Topic:", item["topic"])
    print("Content:", item["content"])
    print("-" * 80)

Score: 0.7248
Title: Weaviate Cloud Authentication
Topic: cloud
Content: Connect to Weaviate Cloud using a cluster URL, Weaviate API key and provider headers such as the OpenAI API key.
--------------------------------------------------------------------------------
Score: 0.6998
Title: Focused RAG over Notebooks
Topic: rag
Content: Focused RAG works best when the collection contains clean chunks from relevant notebooks and Markdown concept notes instead of unrelated files.
--------------------------------------------------------------------------------
Score: 0.6264
Title: Multi-Tenancy
Topic: multi_tenancy
Content: Multi-tenancy isolates data for multiple clients, companies or users inside one shared collection schema.
--------------------------------------------------------------------------------
Score: 0.6161
Title: Hybrid Search with Alpha
Topic: hybrid_search
Content: Hybrid search combines BM25 keyword search with semantic vector search. The alpha parameter controls the balance

In [18]:
CONTEXT_COLLECTION_NAME = "RerankedAnswerContext"

if client.collections.exists(CONTEXT_COLLECTION_NAME):
    client.collections.delete(CONTEXT_COLLECTION_NAME)

answer_contexts = client.collections.create(
    name=CONTEXT_COLLECTION_NAME,
    vector_config=wvc.config.Configure.Vectors.text2vec_openai(
        model="text-embedding-3-small",
    ),
    generative_config=wvc.config.Configure.Generative.openai(
        model="gpt-4o-mini",
    ),
    properties=[
        wvc.config.Property(
            name="question",
            data_type=wvc.config.DataType.TEXT,
        ),
        wvc.config.Property(
            name="context",
            data_type=wvc.config.DataType.TEXT,
        ),
    ],
)

print("Created collection:", CONTEXT_COLLECTION_NAME)

Created collection: RerankedAnswerContext


In [19]:
def build_reranked_context(items: list[dict[str, Any]]) -> str:
    lines = []

    for number, item in enumerate(items, start=1):
        lines.extend(
            [
                f"Result {number}",
                f"Title: {item['title']}",
                f"Topic: {item['topic']}",
                f"Difficulty: {item['difficulty']}",
                f"Priority: {item['priority']}",
                f"Quality score: {item['quality_score']}",
                f"Production ready: {item['production_ready']}",
                f"Reading time: {item['reading_time_minutes']} minutes",
                f"Rerank score: {item['rerank_score']}",
                f"Content: {item['content']}",
                "",
            ]
        )

    return "\n".join(lines)

In [20]:
def answer_with_reranking(
    question: str,
    *,
    candidate_limit: int = 10,
    final_limit: int = 4,
) -> None:
    selected = two_stage_search(
        question,
        candidate_limit=candidate_limit,
        final_limit=final_limit,
        only_production_ready=True,
        reference_date=dt(2026, 5, 25),
    )

    context = build_reranked_context(selected)

    answer_contexts.data.insert(
        {
            "question": question,
            "context": context,
        }
    )

    response = answer_contexts.generate.near_text(
        query=question,
        single_prompt=(
            "Answer the question using only this reranked context.\n\n"
            "Question: {question}\n\n"
            "Context:\n{context}\n\n"
            "Explain which concepts are most relevant and why. "
            "Mention that the answer is based on reranked results."
        ),
        limit=1,
        return_properties=[
            "question",
            "context",
        ],
    )

    print("QUESTION:")
    print(question)

    print("\nSELECTED RERANKED SOURCES:")
    for item in selected:
        print(
            "-",
            item["title"],
            "| score:",
            item["rerank_score"],
            "| topic:",
            item["topic"],
        )

    print("\nANSWER:")
    for obj in response.objects:
        print(obj.generated)

In [21]:
answer_with_reranking(
    "Which Weaviate topics should I learn first to build a production RAG application?"
)

QUESTION:
Which Weaviate topics should I learn first to build a production RAG application?

SELECTED RERANKED SOURCES:
- Focused RAG over Notebooks | score: 0.7479 | topic: rag
- Weaviate Cloud Authentication | score: 0.7205 | topic: cloud
- Generative Search | score: 0.6243 | topic: generative_search
- Multi-Tenancy | score: 0.6229 | topic: multi_tenancy

ANSWER:


In [22]:
answer_with_reranking(
    "Which Weaviate topics are most useful for SaaS architecture with isolated customer data?"
)

QUESTION:
Which Weaviate topics are most useful for SaaS architecture with isolated customer data?

SELECTED RERANKED SOURCES:
- Weaviate Cloud Authentication | score: 0.7342 | topic: cloud
- Multi-Tenancy | score: 0.6982 | topic: multi_tenancy
- Focused RAG over Notebooks | score: 0.6215 | topic: rag
- Hybrid Search with Alpha | score: 0.6128 | topic: hybrid_search

ANSWER:
Based on the reranked results, here are the most relevant Weaviate topics to learn first for building a production RAG (Retrieval-Augmented Generation) application:

1. **RAG (Result 1)**: This topic is fundamental to your application as it addresses the core mechanics of retrieval and generation in a focused manner. RAG systems are optimized when they utilize clean and relevant data, making this an advanced but crucial concept for creating an effective production application. The fact that this topic is marked as production-ready and has a high-quality score underscores its importance.

2. **Generative Search (Res

In [23]:
def two_stage_hybrid_search(
    query: str,
    *,
    alpha: float = 0.5,
    candidate_limit: int = 8,
    final_limit: int = 3,
    reference_date: datetime,
) -> list[dict[str, Any]]:
    response = articles.query.hybrid(
        query=query,
        alpha=alpha,
        limit=candidate_limit,
        return_properties=[
            "title",
            "content",
            "topic",
            "difficulty",
            "priority",
            "reading_time_minutes",
            "quality_score",
            "production_ready",
            "updated_at",
        ],
        return_metadata=MetadataQuery(score=True),
    )

    ranked = []

    for obj in response.objects:
        props = obj.properties

        hybrid_score = obj.metadata.score or 0.0
        priority_score = props["priority"] / 5
        quality_score = props["quality_score"] / 10
        production_score = 1.0 if props["production_ready"] else 0.0
        freshness = freshness_score(
            props["updated_at"],
            reference_date=reference_date,
        )

        final_score = (
            hybrid_score * 0.50
            + priority_score * 0.20
            + quality_score * 0.15
            + production_score * 0.10
            + freshness * 0.05
        )

        ranked.append(
            {
                "uuid": obj.uuid,
                "title": props["title"],
                "content": props["content"],
                "topic": props["topic"],
                "difficulty": props["difficulty"],
                "priority": props["priority"],
                "reading_time_minutes": props["reading_time_minutes"],
                "quality_score": props["quality_score"],
                "production_ready": props["production_ready"],
                "updated_at": props["updated_at"],
                "hybrid_score": round(hybrid_score, 4),
                "rerank_score": round(final_score, 4),
            }
        )

    ranked = sorted(
        ranked,
        key=lambda item: item["rerank_score"],
        reverse=True,
    )

    return ranked[:final_limit]

In [24]:
hybrid_results = two_stage_hybrid_search(
    "RAG BM25 vector search cloud",
    alpha=0.5,
    candidate_limit=10,
    final_limit=5,
    reference_date=dt(2026, 5, 25),
)

for item in hybrid_results:
    print("Rerank score:", item["rerank_score"])
    print("Hybrid score:", item["hybrid_score"])
    print("Title:", item["title"])
    print("Topic:", item["topic"])
    print("Priority:", item["priority"])
    print("Quality:", item["quality_score"])
    print("-" * 80)

Rerank score: 0.9084
Hybrid score: 0.8578
Title: Hybrid Search with Alpha
Topic: hybrid_search
Priority: 5
Quality: 9.3
--------------------------------------------------------------------------------
Rerank score: 0.8721
Hybrid score: 0.9002
Title: Vector Search Basics
Topic: vector_search
Priority: 4
Quality: 8.8
--------------------------------------------------------------------------------
Rerank score: 0.8089
Hybrid score: 0.8857
Title: BM25 Keyword Search
Topic: bm25
Priority: 3
Quality: 8.4
--------------------------------------------------------------------------------
Rerank score: 0.7718
Hybrid score: 0.5526
Title: Focused RAG over Notebooks
Topic: rag
Priority: 5
Quality: 9.7
--------------------------------------------------------------------------------
Rerank score: 0.7462
Hybrid score: 0.5363
Title: Weaviate Cloud Authentication
Topic: cloud
Priority: 5
Quality: 9.2
--------------------------------------------------------------------------------


In [25]:
client.close()